In [0]:
# =============================================================
# NOTEBOOK 01 — BRONZE INGESTION
# Quick Commerce Dark Store Intelligence System
# Layer: Bronze (Raw Ingestion)
# Source: Instacart Dataset from /Volumes/workspace/default/instacart/
# =============================================================

In [0]:
# Define source and target paths
VOLUME_PATH = "/Volumes/workspace/default/instacart/"
BRONZE_DB = "bronze_instacart"

# Create Bronze database if not exists
spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_DB}")
print(f"✅ Database '{BRONZE_DB}' ready")

✅ Database 'bronze_instacart' ready


In [0]:
import os

files = ["orders.csv", "order_products__prior.csv", 
         "order_products__train.csv", "products.csv", 
         "aisles.csv", "departments.csv"]

for f in files:
    path = VOLUME_PATH + f
    try:
        df = spark.read.csv(path, header=True)
        print(f"✅ {f} — accessible")
    except:
        print(f"❌ {f} — NOT found")

✅ orders.csv — accessible
✅ order_products__prior.csv — accessible
✅ order_products__train.csv — accessible
✅ products.csv — accessible
✅ aisles.csv — accessible
✅ departments.csv — accessible


In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Define all files and their table names
tables_config = [
    ("aisles.csv",                  "aisles"),
    ("departments.csv",             "departments"),
    ("products.csv",                "products"),
    ("orders.csv",                  "orders"),
    ("order_products__train.csv",   "order_products_train"),
    ("order_products__prior.csv",   "order_products_prior"),
]

# Loop through and ingest each one
for file_name, table_name in tables_config:
    print(f"⏳ Ingesting {file_name}...")
    
    df = spark.read.csv(VOLUME_PATH + file_name, header=True, inferSchema=True)
    df = df.withColumn("ingested_at", current_timestamp()) \
           .withColumn("source_file", lit(file_name))
    
    df.write.format("delta").mode("overwrite") \
      .saveAsTable(f"{BRONZE_DB}.{table_name}")
    
    print(f"✅ {table_name} → {df.count():,} rows")

print("\n🏆 Bronze Layer Complete!")

⏳ Ingesting aisles.csv...
✅ aisles → 134 rows
⏳ Ingesting departments.csv...
✅ departments → 21 rows
⏳ Ingesting products.csv...
✅ products → 49,688 rows
⏳ Ingesting orders.csv...
✅ orders → 3,421,083 rows
⏳ Ingesting order_products__train.csv...
✅ order_products_train → 1,384,617 rows
⏳ Ingesting order_products__prior.csv...
✅ order_products_prior → 32,434,489 rows

🏆 Bronze Layer Complete!


In [0]:
%sql
select * from bronze_instacart.aisles

aisle_id,aisle,ingested_at,source_file
1,prepared soups salads,2026-03-09T05:49:08.798Z,aisles.csv
2,specialty cheeses,2026-03-09T05:49:08.798Z,aisles.csv
3,energy granola bars,2026-03-09T05:49:08.798Z,aisles.csv
4,instant foods,2026-03-09T05:49:08.798Z,aisles.csv
5,marinades meat preparation,2026-03-09T05:49:08.798Z,aisles.csv
6,other,2026-03-09T05:49:08.798Z,aisles.csv
7,packaged meat,2026-03-09T05:49:08.798Z,aisles.csv
8,bakery desserts,2026-03-09T05:49:08.798Z,aisles.csv
9,pasta sauce,2026-03-09T05:49:08.798Z,aisles.csv
10,kitchen supplies,2026-03-09T05:49:08.798Z,aisles.csv


In [0]:
# ============================================================
# OPTIMIZE — Compact small files for faster queries
# Delta Lake best practice
# ============================================================

tables_to_optimize = [
    "orders",
    "order_products_prior", 
    "order_products_train"
]

for table in tables_to_optimize:
    print(f"⏳ Optimizing {table}...")
    spark.sql(f"OPTIMIZE {BRONZE_DB}.{table}")
    print(f"✅ {table} optimized")

print("\n🏆 OPTIMIZE Complete!")

⏳ Optimizing orders...
✅ orders optimized
⏳ Optimizing order_products_prior...
✅ order_products_prior optimized
⏳ Optimizing order_products_train...
✅ order_products_train optimized

🏆 OPTIMIZE Complete!
